In [1]:
import sys
from pathlib import Path

# Add project root to Python path
PROJECT_ROOT = Path.cwd().parent   # because notebook is in /notebooks
sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src").exists())

Project root: /Users/macbookair/Documents/ai-llm-langchain-sprint
src exists: True


In [2]:
import os
from pathlib import Path

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader

from langchain_ollama import OllamaEmbeddings
from langchain_postgres import PGVector

from src.llm import get_llm

embeddings = OllamaEmbeddings(model="nomic-embed-text")

PG_CONN = "postgresql+psycopg://localhost:5432/ragdb"
COLLECTION = "docs_demo"

In [3]:
docs_dir = Path("../data/docs")
paths = sorted(docs_dir.glob("*.txt"))

docs = []
for p in paths:
    docs.extend(TextLoader(str(p), encoding="utf-8").load())

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80)
chunks = splitter.split_documents(docs)

len(paths), len(docs), len(chunks)

(2, 2, 2)

In [4]:
vs = PGVector(
    connection=PG_CONN,
    collection_name=COLLECTION,
    embeddings=embeddings,
)
"reloaded pgvector collection"

'reloaded pgvector collection'

In [5]:
llm = get_llm(model="llama3.2:3b")

prompt = ChatPromptTemplate.from_template(
    """Answer using ONLY the context. If not in context, say: "I don't know based on the context."

Question: {question}

Context:
{context}
"""
)

retriever = vs.as_retriever(search_kwargs={"k": 4})

def rag_answer(question: str) -> str:
    retrieved = retriever.invoke(question)   # <-- new API
    context = "\n\n".join([f"- {d.page_content}" for d in retrieved])
    result = (prompt | llm | StrOutputParser()).invoke({"question": question, "context": context})
    return result.strip()

In [11]:
def rag_answer_with_citations(question: str, k: int = 4):
    retrieved = vs.as_retriever(search_kwargs={"k": k}).invoke(question)

    citations = []
    for d in retrieved:
        src = d.metadata.get("source", "unknown")
        snip = d.page_content[:180].replace("\n", " ")
        citations.append({"source": src, "snippet": snip})

    context = "\n\n".join([f"- {d.page_content}" for d in retrieved])
    answer = (prompt | llm | StrOutputParser()).invoke({"question": question, "context": context}).strip()

    return {"question": question, "answer": answer, "citations": citations}

rag_answer_with_citations("What databases are mentioned in the docs?", k=4)

{'question': 'What databases are mentioned in the docs?',
 'answer': "I don't know based on the context.",
 'citations': [{'source': '../data/docs/atf_notes.txt',
   'snippet': 'ATF work highlights: - Modernized legacy Java apps using Ant builds, JBoss deployments, and REST APIs. - Worked on data flows involving PostgreSQL and MSSQL. - Focused on debugging'},
  {'source': '../data/docs/llm_notes.txt',
   'snippet': 'LLM notes: - Temperature controls randomness (0 = more deterministic). - Structured outputs (JSON) make LLM responses usable in software systems. - RAG reduces hallucinations by re'}]}

In [7]:
def rag_answer_with_citations(question: str, k: int = 4):
    # retrieve
    retrieved = vs.as_retriever(search_kwargs={"k": k}).invoke(question)

    # build context for the LLM
    context = "\n\n".join([f"- {d.page_content}" for d in retrieved])

    # run LLM
    answer = (prompt | llm | StrOutputParser()).invoke(
        {"question": question, "context": context}
    ).strip()

    if not answer:
        answer = "I don't know based on the context."

    # citations (source + short snippet)
    citations = []
    for d in retrieved:
        src = d.metadata.get("source", "unknown")
        snip = d.page_content[:200].replace("\n", " ")
        citations.append({"source": src, "snippet": snip})

    return {
        "question": question,
        "answer": answer,
        "k": k,
        "citations": citations,
    }

In [9]:
rag_answer("What databases are mentioned in the docs?")

"I don't know based on the context."

In [10]:
rag_answer_with_citations("What databases are mentioned in the docs?", k=4)

{'question': 'What databases are mentioned in the docs?',
 'answer': "I don't know based on the context.",
 'k': 4,
 'citations': [{'source': '../data/docs/atf_notes.txt',
   'snippet': 'ATF work highlights: - Modernized legacy Java apps using Ant builds, JBoss deployments, and REST APIs. - Worked on data flows involving PostgreSQL and MSSQL. - Focused on debugging, production triage,'},
  {'source': '../data/docs/llm_notes.txt',
   'snippet': 'LLM notes: - Temperature controls randomness (0 = more deterministic). - Structured outputs (JSON) make LLM responses usable in software systems. - RAG reduces hallucinations by retrieving relevant co'}]}